# 第 28 课：FastAPI HTTP 推理服务——模型契约、健康检查与错误边界

HTTP 适合单块、离线短请求或控制接口；持续音频流更适合下一课的 WebSocket。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 量化与部署 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 27 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | HTTP contract、health/readiness、无状态 cache |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：HTTP contract、health/readiness、无状态 cache。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：PyTorch 基线输出；shape/dtype/dynamic axes；流式 cache；P50/P99 与 RTF。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](27_ONNXRuntime_INT8量化与Benchmark.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：冻结模型与数值基线、目标硬件、输入输出/状态协议
  ↓ 本课要学会的变换、状态或判断
输出：有一致性、性能、并发隔离和回滚证据的部署产物
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import time
import sys
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT=find_root(); ARTIFACTS=ROOT/"artifacts";ARTIFACTS.mkdir(exist_ok=True)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

from fastapi.testclient import TestClient
from deployment.app import app,MODEL_PATH
from deployment.model import FEATURE_DIM,CHUNK_FRAMES,CACHE_FRAMES
client=TestClient(app)
print("loaded model:",MODEL_PATH)

## 1. 健康检查不是准确率检查

`/health` 证明进程能够加载模型并响应；readiness 还应确认必要资源已就绪。模型质量需要单独的回归集。

In [ ]:
r=client.get("/health");print(r.status_code,r.json())

## 2. 一次显式 cache 的推理请求

In [ ]:
rng=np.random.default_rng(1);frames=rng.normal(size=(CHUNK_FRAMES,FEATURE_DIM)).astype(np.float32)
r=client.post("/infer",json={"frames":frames.tolist()})
body=r.json();print("status",r.status_code,"logits shape",np.asarray(body["logits"]).shape,"cache shape",np.asarray(body["cache"]).shape)

## 3. 客户端负责把 cache 传回去

In [ ]:
cache=body["cache"]
frames2=rng.normal(size=(CHUNK_FRAMES,FEATURE_DIM)).astype(np.float32)
r2=client.post("/infer",json={"frames":frames2.tolist(),"cache":cache})
print(r2.status_code,np.asarray(r2.json()["logits"]).shape)

HTTP 显式 cache 的优点是服务端无会话状态，容易横向扩容；缺点是 cache 反复传输、客户端复杂、容易篡改。WebSocket 可以把 cache 留在连接内。

## 4. 启动真实服务

```powershell
uv run uvicorn deployment.app:app --host 127.0.0.1 --port 8000
```

生产环境还需要请求大小限制、超时、认证、TLS、日志脱敏、进程管理和滚动升级。

## 本课测试

1. `/health` 返回 200 是否代表模型准确？
2. 无状态 HTTP 为什么容易扩容？
3. 显式传输 cache 有什么风险？
4. 为什么不能把任意长度 JSON 一次读入内存？
5. 模型版本应在哪里暴露？

<details><summary>展开参考答案</summary>

1. 不代表。2. 任意实例都能处理下一请求。3. 带宽、客户端状态错误和篡改。4. 可能造成内存/拒绝服务风险。5. health/metadata、日志和每个结果事件中均可包含可追踪版本。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 28 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `HTTP contract`、`health/readiness`、`无状态 cache`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**客户端传回错误 shape/cache**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**为 /infer 添加输入失败测试和模型版本**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**对比 HTTP 与下一课 WebSocket 状态**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：HTTP contract、health/readiness、无状态 cache。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 HTTP contract、health/readiness、无状态 cache。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
